# Long-Time Fisher-KPP Time Integrator Comparison

This notebook compares forward Euler, backward Euler, trapezoidal, and RK4 on the same 1D Fisher-KPP method-of-lines problem through `t=30`.

The reference image shows a long-time surface and a time trace. For the Fisher-KPP equation used here, an oscillatory `rho(t)` is not physically expected because the scalar parabolic problem obeys a maximum-principle style bound. The comparable long-time diagnostics are the moving front, mean density, and a fixed spatial probe `rho(t)=u(x_probe,t)`.


## 1. Setup


In [ ]:
%matplotlib inline

from __future__ import annotations

import csv
import subprocess
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

REPO_URL = "https://github.com/rladbsco24/fisher-pinn.git"
REPO_BRANCH = "main"


def _has_rk4_project(root: Path) -> bool:
    return (root / "fisher-kpp-rk4" / "src" / "fisher_kpp_rk4").exists()


def _prepare_colab_repo(repo_dir: Path) -> Path:
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
    elif (repo_dir / ".git").exists():
        subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo_dir, check=True)
        subprocess.run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=repo_dir, check=True)
    return repo_dir.resolve()


PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if _has_rk4_project(candidate):
        PROJECT_ROOT = candidate
        break

if not _has_rk4_project(PROJECT_ROOT) and Path("/content").exists():
    PROJECT_ROOT = _prepare_colab_repo(Path("/content/fisher-pinn"))

if not _has_rk4_project(PROJECT_ROOT):
    raise RuntimeError("Run this notebook from the fisher-pinn repository or use Colab with network access.")

RK4_ROOT = PROJECT_ROOT / "fisher-kpp-rk4"
SRC_DIR = RK4_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

OUTPUT_DIR = RK4_ROOT / "outputs" / "long_time_methods_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"project root: {PROJECT_ROOT}")
print(f"output dir:   {OUTPUT_DIR}")


## 2. Fair Long-Time Parameters


In [ ]:
from fisher_kpp_rk4 import check_forward_euler_stability, check_rk4_stability, relative_l2, solve_1d_method
from fisher_kpp_rk4.config import (
    LONG_TIME_D,
    LONG_TIME_DT,
    LONG_TIME_DX,
    LONG_TIME_L,
    LONG_TIME_LEFT_BC,
    LONG_TIME_NT,
    LONG_TIME_NX,
    LONG_TIME_PROBE_X,
    LONG_TIME_R,
    LONG_TIME_RIGHT_BC,
    LONG_TIME_SAVE_INTERVAL,
    LONG_TIME_T,
    LONG_TIME_X,
    long_time_initial_condition,
)

METHODS = ("forward_euler", "backward_euler", "trapezoidal", "rk4")
fe_info = check_forward_euler_stability(LONG_TIME_DX, LONG_TIME_DT, LONG_TIME_D, LONG_TIME_R, dim=1)
rk4_info = check_rk4_stability(LONG_TIME_DX, LONG_TIME_DT, LONG_TIME_D, LONG_TIME_R, dim=1)
print(f"D={LONG_TIME_D}, r={LONG_TIME_R}, L={LONG_TIME_L}, T={LONG_TIME_T}")
print(f"Nx={LONG_TIME_NX}, dx={LONG_TIME_DX:.6g}, Nt={LONG_TIME_NT}, dt={LONG_TIME_DT:.6g}")
print(f"Forward Euler safe: {fe_info['is_practically_safe']} (limit={fe_info['dt_practical']:.6g})")
print(f"RK4 safe:           {rk4_info['is_practically_safe']} (limit={rk4_info['dt_practical']:.6g})")


## 3. Run All Four Methods


In [ ]:
results = {}
for method in METHODS:
    results[method] = solve_1d_method(
        method,
        x=LONG_TIME_X,
        dt=LONG_TIME_DT,
        Nt=LONG_TIME_NT,
        D=LONG_TIME_D,
        r=LONG_TIME_R,
        initial_condition=long_time_initial_condition,
        left_bc=LONG_TIME_LEFT_BC,
        right_bc=LONG_TIME_RIGHT_BC,
        save_interval=LONG_TIME_SAVE_INTERVAL,
        probe_x=LONG_TIME_PROBE_X,
    )
    print(method, results[method]["snapshots"].shape)


## 4. Metrics


In [ ]:
rk4_final = results["rk4"]["u_final"]
rows = []
for method, result in results.items():
    rows.append(
        {
            "method": method,
            "final_front": float(result["fronts"][-1]),
            "final_mass": float(result["mass"][-1]),
            "final_rho": float(result["rho"][-1]),
            "relative_l2_to_rk4_final": 0.0 if method == "rk4" else relative_l2(result["u_final"], rk4_final),
            "max_newton_iterations": int(result["newton_iterations"].max()) if len(result["newton_iterations"]) else 0,
        }
    )

md_table = "| method | final front | final mass | final rho | final L2 vs RK4 | max Newton iters |\n"
md_table += "|---|---:|---:|---:|---:|---:|\n"
for row in rows:
    md_table += (
        f"| {row['method']} | {row['final_front']:.4f} | {row['final_mass']:.4f} | "
        f"{row['final_rho']:.4f} | {row['relative_l2_to_rk4_final']:.3e} | {row['max_newton_iterations']} |\n"
    )
display(Markdown(md_table))


## 5. Visualize Long-Time Trends


In [ ]:
import matplotlib.pyplot as plt

def savefig(name: str):
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print(path)

rk4 = results["rk4"]
tt, xx = np.meshgrid(rk4["times"], rk4["x"], indexing="ij")
fig = plt.figure(figsize=(9, 6.2))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(tt, xx, rk4["snapshots"], cmap="viridis", linewidth=0.15, edgecolor="#343a40", alpha=0.92)
ax.set_xlabel("time")
ax.set_ylabel("space")
ax.set_zlabel("u")
ax.set_title("Adjusted RK4 long-time Fisher-KPP surface")
fig.tight_layout()
savefig("adjusted_rk4_long_time_surface.png")
plt.show()

plt.figure(figsize=(8.5, 5.2))
for method, result in results.items():
    plt.plot(result["times"], result["rho"], label=method)
plt.xlabel("time")
plt.ylabel(f"rho(t)=u(x={LONG_TIME_PROBE_X:g},t)")
plt.title("Long-time probe trend")
plt.grid(alpha=0.25)
plt.legend()
savefig("long_time_probe_rho.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for method, result in results.items():
    axes[0].plot(result["times"], result["fronts"], label=method)
    axes[1].plot(result["times"], result["mass"], label=method)
axes[0].set_title("u=0.5 front position")
axes[0].set_xlabel("time")
axes[0].set_ylabel("x")
axes[0].grid(alpha=0.25)
axes[1].set_title("mean density")
axes[1].set_xlabel("time")
axes[1].set_ylabel("mean u")
axes[1].grid(alpha=0.25)
axes[1].legend()
fig.savefig(OUTPUT_DIR / "long_time_front_mass.png", dpi=180)
plt.show()


## 6. Save Numerical Outputs


In [ ]:
with (OUTPUT_DIR / "long_time_method_summary.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

payload = {
    "x": LONG_TIME_X,
    "times": results["rk4"]["times"],
    "D": np.array(LONG_TIME_D),
    "r": np.array(LONG_TIME_R),
    "L": np.array(LONG_TIME_L),
    "T": np.array(LONG_TIME_T),
    "dt": np.array(LONG_TIME_DT),
    "dx": np.array(LONG_TIME_DX),
    "probe_x": np.array(LONG_TIME_PROBE_X),
}
for method, result in results.items():
    for key in ("snapshots", "fronts", "mass", "rho", "u_final"):
        payload[f"{method}_{key}"] = result[key]
np.savez(OUTPUT_DIR / "long_time_method_results.npz", **payload)
print("Saved outputs under", OUTPUT_DIR)
